# ParaView + trame in Jupyter

Use the **ParaView 6.1.1 osmesa (pvpython + trame)** kernel. Restart the kernel before rerunning everything: the trame server keeps running between cells.

Check that the trame Jupyter extension set up the kernel. You want `jupyter-extension jupyter`. If you get `None None`, this JupyterLab isn't the one from `~/TRAME/bin` (see the README).

In [ ]:
import os
print(os.environ.get("TRAME_IFRAME_BUILDER"), os.environ.get("TRAME_BACKEND"))

A ParaView pipeline shown in a trame app. `VtkRemoteView` renders on the node (OSMesa) and streams images; dragging rotates, Shift-drag pans, scroll zooms.

In [ ]:
from paraview import simple
from trame.app import get_server
from trame.ui.vuetify3 import SinglePageLayout
from trame.widgets import paraview as pvw, vuetify3 as v3

server = get_server("pv", client_type="vue3")

cone = simple.Cone()
simple.Show(cone)
view = simple.Render()
simple.ResetCamera()
print(view.GetRenderWindow().GetClassName())   # want: vtkOSOpenGLRenderWindow

with SinglePageLayout(server) as layout:
    layout.title.set_text("ParaView in Jupyter")
    with layout.content:
        # fill-height parent, or the view can end up 0 px tall
        with v3.VContainer(fluid=True, classes="pa-0 fill-height"):
            pv_view = pvw.VtkRemoteView(view, interactive_ratio=1)

# Starts the trame server in the kernel's event loop and shows the app
await layout.display_cell(height=600)

Optional: spin the camera. Set `spinning = False` (next cell) to stop.

In [ ]:
import asyncio

spinning = True

async def spin(step=2, delay=0.05):
    while spinning:
        view.GetActiveCamera().Azimuth(step)
        pv_view.update()
        await asyncio.sleep(delay)

spin_task = asyncio.ensure_future(spin())

In [ ]:
spinning = False